In [ ]:
# ============================================
# UVFD Video-Language Forensic Agent
# CPU Prototype
# ============================================

from pathlib import Path
import cv2
import pandas as pd
import numpy as np
import random
import os

# Reproducibility
random.seed(42)
np.random.seed(42)

# Configuration
VIDEOS_PER_CLASS = 3
MAX_RESULTS = 4

# Kaggle dataset location
DATASET_ROOT = Path("/kaggle/input/datasets")

print("Setup complete.")
print("Dataset root:", DATASET_ROOT)
print("Videos per class:", VIDEOS_PER_CLASS)
print("Maximum query results:", MAX_RESULTS)

In [ ]:
# Find the training directory automatically

train_dirs = [
    p for p in DATASET_ROOT.rglob("train")
    if p.is_dir()
]

print("Training directories found:")

for path in train_dirs:
    print(path)

if not train_dirs:
    raise FileNotFoundError("UVFD train folder was not found.")

TRAIN_DIR = train_dirs[0]

print("\nUsing:")
print(TRAIN_DIR)

In [ ]:
# Find class folders

class_folders = [
    p for p in TRAIN_DIR.iterdir()
    if p.is_dir()
]

print("Classes found:\n")

for folder in class_folders:
    print(folder.name)

In [ ]:
# Supported video formats
VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm",
    ".mpeg",
    ".mpg"
}

prototype_videos = []

for class_name in ["Fight", "NonFight"]:

    class_dir = TRAIN_DIR / class_name

    videos = [
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    ]

    if len(videos) < VIDEOS_PER_CLASS:
        raise ValueError(
            f"Not enough videos in {class_name}. "
            f"Found {len(videos)}."
        )

    selected = random.sample(videos, VIDEOS_PER_CLASS)

    for video_path in selected:
        prototype_videos.append({
            "video_path": str(video_path),
            "video_name": video_path.name,
            "label": class_name
        })

print("Selected prototype videos:\n")

for i, video in enumerate(prototype_videos, 1):
    print(
        f"{i}. {video['label']:8s} | "
        f"{video['video_name']}"
    )

print("\nTotal videos:", len(prototype_videos))

In [ ]:
# ============================================
# Cell 5 — Build Video Metadata Index
# ============================================

metadata = []

for video in prototype_videos:

    cap = cv2.VideoCapture(video["video_path"])

    if not cap.isOpened():
        print("Could not open:", video["video_name"])
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if fps and fps > 0:
        duration = frame_count / fps
    else:
        duration = 0

    metadata.append({
        "video_name": video["video_name"],
        "video_path": video["video_path"],
        "label": video["label"],
        "fps": round(fps, 2),
        "frame_count": frame_count,
        "duration": round(duration, 2),
        "width": width,
        "height": height
    })

    cap.release()

metadata_df = pd.DataFrame(metadata)

print("Video Metadata Index:")
display(metadata_df)

print("\nTotal videos indexed:", len(metadata_df))
print("Total duration:", round(metadata_df["duration"].sum(), 2), "seconds")

In [ ]:
# ============================================
# Cell 6 — Create Timestamped Video Segments
# ============================================

SEGMENT_DURATION = 4  # seconds

segments = []

for _, row in metadata_df.iterrows():

    duration = row["duration"]

    start_time = 0

    while start_time < duration:

        end_time = min(
            start_time + SEGMENT_DURATION,
            duration
        )

        segments.append({
            "video_name": row["video_name"],
            "video_path": row["video_path"],
            "label": row["label"],
            "start_time": round(start_time, 2),
            "end_time": round(end_time, 2)
        })

        start_time += SEGMENT_DURATION


segments_df = pd.DataFrame(segments)

print("Timestamped segments created:")
display(segments_df)

print("\nTotal segments:", len(segments_df))
print("Videos:", segments_df["video_name"].nunique())

In [ ]:
# ============================================
# Cell 7 — Extract Motion Features
# ============================================

def calculate_motion_score(video_path, start_time, end_time, sample_count=8):
    
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return 0.0

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        cap.release()
        return 0.0

    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)

    if end_frame <= start_frame:
        cap.release()
        return 0.0

    frame_indices = np.linspace(
        start_frame,
        end_frame,
        sample_count,
        dtype=int
    )

    previous_frame = None
    motion_values = []

    for frame_index in frame_indices:

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))

        success, frame = cap.read()

        if not success:
            continue

        # Resize for faster CPU processing
        frame = cv2.resize(frame, (160, 90))

        # Convert to grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if previous_frame is not None:

            difference = cv2.absdiff(
                previous_frame,
                gray
            )

            motion = np.mean(difference)

            motion_values.append(float(motion))

        previous_frame = gray

    cap.release()

    if not motion_values:
        return 0.0

    return float(np.mean(motion_values))


# Calculate motion for every segment
motion_scores = []

for i, row in segments_df.iterrows():

    score = calculate_motion_score(
        row["video_path"],
        row["start_time"],
        row["end_time"]
    )

    motion_scores.append(score)

    print(
        f"{i + 1}/{len(segments_df)} "
        f"| {row['video_name']} "
        f"| {row['start_time']:.2f}-{row['end_time']:.2f}s "
        f"| Motion: {score:.2f}"
    )


segments_df["motion_score"] = motion_scores

print("\nMotion feature extraction complete.")

display(segments_df)

In [ ]:
# ============================================
# Cell 8 — Normalize Motion Scores
# ============================================

min_motion = segments_df["motion_score"].min()
max_motion = segments_df["motion_score"].max()

if max_motion > min_motion:
    segments_df["motion_normalized"] = (
        (segments_df["motion_score"] - min_motion)
        / (max_motion - min_motion)
    )
else:
    segments_df["motion_normalized"] = 0.0

print("Motion normalization complete.\n")

display(
    segments_df[
        [
            "video_name",
            "label",
            "start_time",
            "end_time",
            "motion_score",
            "motion_normalized"
        ]
    ]
)

In [ ]:
# ============================================
# Cell 9 — Create Semantic Descriptions
# ============================================

def describe_segment(row):
    
    motion = row["motion_normalized"]

    if motion >= 0.70:
        motion_text = "high motion activity"
    elif motion >= 0.30:
        motion_text = "moderate motion activity"
    else:
        motion_text = "low motion activity"

    if row["label"] == "Fight":
        event_text = "fight or violent activity"
    else:
        event_text = "non-fight activity"

    return (
        f"{event_text}, "
        f"{motion_text}, "
        f"video segment from {row['start_time']:.1f} "
        f"to {row['end_time']:.1f} seconds"
    )


segments_df["description"] = segments_df.apply(
    describe_segment,
    axis=1
)

print("Semantic descriptions created:\n")

display(
    segments_df[
        [
            "video_name",
            "label",
            "start_time",
            "end_time",
            "motion_normalized",
            "description"
        ]
    ]
)

In [ ]:
# ============================================
# Cell 10 — Natural-Language Query Engine
# ============================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create text index
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

text_matrix = vectorizer.fit_transform(
    segments_df["description"]
)

print("Text index created.")
print("Indexed segments:", len(segments_df))


def search_videos(query, max_results=MAX_RESULTS):

    query = query.strip()

    if not query:
        print("Please enter a query.")
        return pd.DataFrame()

    # Convert query into TF-IDF representation
    query_vector = vectorizer.transform([query])

    # Text similarity
    text_scores = cosine_similarity(
        query_vector,
        text_matrix
    )[0]

    results = segments_df.copy()

    results["text_similarity"] = text_scores

    # Combine semantic similarity with visual motion
    results["retrieval_score"] = (
        0.75 * results["text_similarity"]
        + 0.25 * results["motion_normalized"]
    )

    # Sort by relevance
    results = results.sort_values(
        "retrieval_score",
        ascending=False
    )

    # HARD LIMIT: maximum 4 results
    results = results.head(max_results)

    return results[
        [
            "video_name",
            "label",
            "start_time",
            "end_time",
            "motion_score",
            "text_similarity",
            "retrieval_score",
            "description"
        ]
    ].reset_index(drop=True)


print("\nQuery engine ready.")

In [ ]:
# ============================================
# Cell 11 — Test Investigative Queries
# ============================================

test_queries = [
    "violent fight activity",
    "high movement activity",
    "non fight activity",
    "low movement activity"
]

for query in test_queries:

    print("=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = search_videos(query)

    display(results)

In [ ]:
# ============================================
# Cell 12 — Generate Natural-Language Answers
# ============================================

def forensic_answer(query):
    
    results = search_videos(query)

    if results.empty:
        return "I could not find any relevant video segments."

    print(f"Investigation Query: {query}\n")
    print(
        f"I found {len(results)} relevant video segment"
        f"{'s' if len(results) != 1 else ''}.\n"
    )

    for i, row in results.iterrows():

        start = row["start_time"]
        end = row["end_time"]

        print(
            f"{i + 1}. {row['video_name']} "
            f"({row['label']})"
        )

        print(
            f"   Timestamp: {start:.2f}s - {end:.2f}s"
        )

        print(
            f"   Motion score: {row['motion_score']:.2f}"
        )

        print(
            f"   Relevance: {row['retrieval_score']:.3f}"
        )

        print(
            f"   Evidence: {row['description']}"
        )

        print()


# Test conversational response
forensic_answer("violent fight activity")

In [ ]:
# ============================================
# Cell 13 — Create Evidence Clips
# ============================================

from pathlib import Path
import subprocess

EVIDENCE_DIR = Path("/kaggle/working/evidence_clips")
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)


def create_evidence_clip(video_path, start_time, end_time, output_path):

    duration = end_time - start_time

    command = [
        "ffmpeg",
        "-y",
        "-ss", str(start_time),
        "-i", video_path,
        "-t", str(duration),
        "-an",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-pix_fmt", "yuv420p",
        str(output_path)
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    return result.returncode == 0


# ============================================
# Search for relevant segments
# ============================================

query = "violent fight activity"

results = search_videos(query)

evidence_files = []


for i, row in results.iterrows():

    # Get the original video path from metadata
    matching_video = metadata_df[
        metadata_df["video_name"] == row["video_name"]
    ]

    if matching_video.empty:
        print(
            f"Could not find video path for "
            f"{row['video_name']}"
        )
        continue

    video_path = matching_video.iloc[0]["video_path"]

    output_path = (
        EVIDENCE_DIR /
        f"evidence_{i + 1}.mp4"
    )

    success = create_evidence_clip(
        video_path,
        row["start_time"],
        row["end_time"],
        output_path
    )

    if success:

        evidence_files.append(str(output_path))

        print(
            f"Created evidence clip {i + 1}: "
            f"{row['video_name']} "
            f"[{row['start_time']:.2f}s - "
            f"{row['end_time']:.2f}s]"
        )

    else:

        print(
            f"Failed to create clip {i + 1}: "
            f"{row['video_name']}"
        )


print("\nEvidence clips created:", len(evidence_files))
print("Saved to:", EVIDENCE_DIR)


In [ ]:
# ============================================
# Cell 14 — Display Evidence Clips
# ============================================

from IPython.display import Video, display

print("Displaying retrieved evidence:\n")

for i, clip_path in enumerate(evidence_files, 1):

    print(f"Evidence {i}: {clip_path}")

    display(
        Video(
            clip_path,
            embed=True,
            width=640
        )
    )

In [ ]:
# ============================================
# Cell 15 — Improved Forensic Retrieval
# ============================================

def search_videos_v2(query, max_results=MAX_RESULTS):

    query = query.strip().lower()

    if not query:
        print("Please enter a query.")
        return pd.DataFrame()

    # ----------------------------------------
    # Detect investigation intent
    # ----------------------------------------

    violence_keywords = [
        "fight",
        "fighting",
        "violence",
        "violent",
        "attack",
        "assault",
        "brawl",
        "physical confrontation"
    ]

    nonfight_keywords = [
        "non fight",
        "non-fight",
        "normal activity",
        "normal",
        "peaceful"
    ]

    is_violence_query = any(
        word in query
        for word in violence_keywords
    )

    is_nonfight_query = any(
        word in query
        for word in nonfight_keywords
    )

    # ----------------------------------------
    # Select candidate segments
    # ----------------------------------------

    candidates = segments_df.copy()

    if is_violence_query and not is_nonfight_query:

        candidates = candidates[
            candidates["label"] == "Fight"
        ]

    elif is_nonfight_query:

        candidates = candidates[
            candidates["label"] == "NonFight"
        ]

    # ----------------------------------------
    # TF-IDF similarity
    # ----------------------------------------

    candidate_indices = candidates.index.tolist()

    candidate_matrix = text_matrix[candidate_indices]

    query_vector = vectorizer.transform([query])

    text_scores = cosine_similarity(
        query_vector,
        candidate_matrix
    )[0]

    candidates = candidates.copy()

    candidates["text_similarity"] = text_scores

    # ----------------------------------------
    # Retrieval score
    # ----------------------------------------

    candidates["retrieval_score"] = (
        0.80 * candidates["text_similarity"]
        + 0.20 * candidates["motion_normalized"]
    )

    # ----------------------------------------
    # Rank candidates
    # ----------------------------------------

    candidates = candidates.sort_values(
        "retrieval_score",
        ascending=False
    )

    # ----------------------------------------
    # Maximum 1 result per video
    # ----------------------------------------

    candidates = candidates.drop_duplicates(
        subset=["video_name"],
        keep="first"
    )

    # ----------------------------------------
    # HARD LIMIT: maximum 4
    # ----------------------------------------

    candidates = candidates.head(max_results)

    return candidates[
        [
            "video_name",
            "video_path",
            "label",
            "start_time",
            "end_time",
            "motion_score",
            "text_similarity",
            "retrieval_score",
            "description"
        ]
    ].reset_index(drop=True)


print("Improved forensic retrieval engine ready.")

In [ ]:
# ============================================
# Test Improved Retrieval
# ============================================

query = "violent fight activity"

results_v2 = search_videos_v2(query)

print("QUERY:", query)
print()

display(
    results_v2[
        [
            "video_name",
            "label",
            "start_time",
            "end_time",
            "motion_score",
            "retrieval_score"
        ]
    ]
)

In [ ]:
# ============================================
# Cell 16 — Inspect Retrieved Segments
# ============================================

from IPython.display import Image, display
import matplotlib.pyplot as plt

def extract_middle_frame(video_path, start_time, end_time):
    
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    middle_time = (start_time + end_time) / 2

    cap.set(
        cv2.CAP_PROP_POS_MSEC,
        middle_time * 1000
    )

    success, frame = cap.read()

    cap.release()

    if not success:
        return None

    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


print("Visual inspection of retrieved segments:\n")

for i, row in results_v2.iterrows():

    frame = extract_middle_frame(
        row["video_path"],
        row["start_time"],
        row["end_time"]
    )

    if frame is not None:

        plt.figure(figsize=(8, 5))
        plt.imshow(frame)
        plt.title(
            f"Result {i+1} | {row['video_name']} | "
            f"{row['start_time']:.2f}s - {row['end_time']:.2f}s"
        )
        plt.axis("off")
        plt.show()

In [ ]:
# ============================================
# Cell 17 — Validate Retrieved Evidence
# ============================================

def validate_segment(video_path, start_time, end_time):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        cap.release()
        return False

    middle_time = (start_time + end_time) / 2

    cap.set(
        cv2.CAP_PROP_POS_MSEC,
        middle_time * 1000
    )

    success, frame = cap.read()

    cap.release()

    if not success or frame is None:
        return False

    # Reject obviously invalid frames
    if frame.size == 0:
        return False

    height, width = frame.shape[:2]

    if height < 50 or width < 50:
        return False

    # Check whether the frame contains usable variation
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if np.std(gray) < 1:
        return False

    return True


validated_results = []

for _, row in results_v2.iterrows():

    valid = validate_segment(
        row["video_path"],
        row["start_time"],
        row["end_time"]
    )

    if valid:
        validated_results.append(row)

        print(
            f"VALID   | {row['video_name']} | "
            f"{row['start_time']:.2f}-{row['end_time']:.2f}s"
        )

    else:
        print(
            f"INVALID | {row['video_name']} | "
            f"{row['start_time']:.2f}-{row['end_time']:.2f}s"
        )


validated_results_df = pd.DataFrame(validated_results)

print(
    "\nValid evidence segments:",
    len(validated_results_df)
)

display(
    validated_results_df[
        [
            "video_name",
            "label",
            "start_time",
            "end_time",
            "retrieval_score"
        ]
    ]
)

In [ ]:
# ============================================
# Cell 18 — Final Evidence Retrieval
# ============================================

def forensic_search(query, max_results=MAX_RESULTS):

    # Use improved retrieval
    results = search_videos_v2(
        query,
        max_results=len(segments_df)
    )

    if results.empty:
        return results

    valid_results = []

    # Validate each candidate
    for _, row in results.iterrows():

        if validate_segment(
            row["video_path"],
            row["start_time"],
            row["end_time"]
        ):
            valid_results.append(row)

        # Stop once we have enough valid evidence
        if len(valid_results) >= max_results:
            break

    if not valid_results:
        return pd.DataFrame()

    final_results = pd.DataFrame(valid_results)

    return final_results.head(max_results).reset_index(drop=True)


# ============================================
# Test final retrieval
# ============================================

query = "violent fight activity"

final_results = forensic_search(query)

print("QUERY:", query)
print("\nFINAL EVIDENCE RESULTS:\n")

display(
    final_results[
        [
            "video_name",
            "label",
            "start_time",
            "end_time",
            "motion_score",
            "retrieval_score",
            "description"
        ]
    ]
)

In [ ]:
# ============================================
# Cell 19 — Generate Final Evidence Clips
# ============================================

# Clear old evidence clips
for old_file in EVIDENCE_DIR.glob("evidence_*.mp4"):
    old_file.unlink()

final_evidence_files = []

for i, row in final_results.iterrows():

    output_path = (
        EVIDENCE_DIR /
        f"final_evidence_{i + 1}.mp4"
    )

    success = create_evidence_clip(
        row["video_path"],
        row["start_time"],
        row["end_time"],
        output_path
    )

    if success:
        final_evidence_files.append(str(output_path))

        print(
            f"Created final evidence {i + 1}: "
            f"{row['video_name']} | "
            f"{row['start_time']:.2f}s - "
            f"{row['end_time']:.2f}s"
        )
    else:
        print(
            f"Failed: {row['video_name']}"
        )

print(
    "\nFinal evidence clips:",
    len(final_evidence_files)
)

In [ ]:
# ============================================
# Cell 20 — Display Final Evidence
# ============================================

from IPython.display import Video, display

for i, clip_path in enumerate(final_evidence_files, 1):

    row = final_results.iloc[i - 1]

    print(
        f"Evidence {i}: "
        f"{row['video_name']} | "
        f"{row['start_time']:.2f}s - "
        f"{row['end_time']:.2f}s"
    )

    display(
        Video(
            clip_path,
            embed=True,
            width=640
        )
    )

In [ ]:
# ============================================
# Cell 22 — Agent Investigation Response
# ============================================

def generate_agent_response(query, results):
    
    if results.empty:
        return (
            "I could not find any relevant evidence "
            "for this investigation query."
        )

    response = []

    response.append(
        f'Investigation query: "{query}"'
    )

    response.append(
        f"\nI found {len(results)} relevant "
        f"video segment(s) in the indexed footage."
    )

    for i, row in results.iterrows():

        start = row["start_time"]
        end = row["end_time"]

        if row["label"] == "Fight":
            event = "fight or physical confrontation"
        else:
            event = "non-fight activity"

        if row["motion_score"] >= 20:
            activity = "high visual motion"
        elif row["motion_score"] >= 10:
            activity = "moderate visual motion"
        else:
            activity = "low visual motion"

        response.append(
            f"\nEvidence {i + 1}:"
            f"\n- Source: {row['video_name']}"
            f"\n- Timestamp: {start:.2f}s–{end:.2f}s"
            f"\n- Detected category: {event}"
            f"\n- Visual activity: {activity}"
            f"\n- Retrieval relevance: "
            f"{row['retrieval_score']:.3f}"
        )

    response.append(
        "\n\nThe returned timestamps correspond to "
        "the evidence segments retrieved by the "
        "forensic search system."
    )

    return "\n".join(response)


# ============================================
# Test the agent
# ============================================

query = "violent fight activity"

agent_response = generate_agent_response(
    query,
    final_results
)

print(agent_response)

In [ ]:
# ============================================
# Cell 23 — Conversational Forensic Agent
# ============================================

conversation_state = {
    "last_query": None,
    "last_results": pd.DataFrame(),
    "last_response": None
}


def conversational_agent(user_query):

    query = user_query.lower().strip()

    # ----------------------------------------
    # First investigation query
    # ----------------------------------------

    if any(word in query for word in [
        "find",
        "search",
        "show",
        "detect",
        "look for"
    ]):

        # New investigation
        results = forensic_search(user_query)

        conversation_state["last_query"] = user_query
        conversation_state["last_results"] = results

        if results.empty:
            response = (
                "I could not find relevant evidence "
                "for that query."
            )
        else:
            response = generate_agent_response(
                user_query,
                results
            )

        conversation_state["last_response"] = response

        return response

    # ----------------------------------------
    # Follow-up: strongest result
    # ----------------------------------------

    if any(word in query for word in [
        "strongest",
        "best",
        "top",
        "most relevant"
    ]):

        results = conversation_state["last_results"]

        if results.empty:
            return (
                "There is no previous investigation "
                "result to refer to."
            )

        row = results.iloc[0]

        return (
            f"The strongest matching evidence is "
            f"{row['video_name']} from "
            f"{row['start_time']:.2f}s to "
            f"{row['end_time']:.2f}s. "
            f"It has a retrieval relevance score of "
            f"{row['retrieval_score']:.3f}."
        )

    # ----------------------------------------
    # Follow-up: other video
    # ----------------------------------------

    if any(word in query for word in [
        "other",
        "another",
        "next"
    ]):

        results = conversation_state["last_results"]

        if len(results) < 2:
            return (
                "There is no other matching video "
                "in the previous results."
            )

        row = results.iloc[1]

        return (
            f"Another matching video is "
            f"{row['video_name']} from "
            f"{row['start_time']:.2f}s to "
            f"{row['end_time']:.2f}s."
        )

    # ----------------------------------------
    # Follow-up: repeat previous investigation
    # ----------------------------------------

    if any(word in query for word in [
        "repeat",
        "again",
        "previous"
    ]):

        if conversation_state["last_query"] is None:
            return "There is no previous investigation."

        return generate_agent_response(
            conversation_state["last_query"],
            conversation_state["last_results"]
        )

    # ----------------------------------------
    # Unknown follow-up
    # ----------------------------------------

    return (
        "I can help investigate the indexed footage. "
        "You can ask me to find an event, show the "
        "strongest result, or show another matching video."
    )


print("Conversational forensic agent ready.")

In [ ]:
# ============================================
# Cell 24 — Conversational Agent + Evidence
# ============================================

from IPython.display import Video, display


def investigate(query):
    
    print("=" * 80)
    print("🤖 FORENSIC CCTV AGENT")
    print("=" * 80)

    print(f"\n🔎 Investigator Query:")
    print(f'"{query}"\n')

    # Run conversational agent
    response = conversational_agent(query)

    print("🤖 Agent:")
    print(response)

    # ----------------------------------------
    # Show evidence for a new investigation
    # ----------------------------------------

    if conversation_state["last_results"].empty:
        return

    results = conversation_state["last_results"]

    print("\n" + "=" * 80)
    print("🎥 RETRIEVED EVIDENCE")
    print("=" * 80)

    displayed = 0

    for i, row in results.iterrows():

        if displayed >= MAX_RESULTS:
            break

        # Create evidence clip
        output_path = (
            EVIDENCE_DIR /
            f"conversation_evidence_{displayed + 1}.mp4"
        )

        success = create_evidence_clip(
            row["video_path"],
            row["start_time"],
            row["end_time"],
            output_path
        )

        if not success:
            continue

        print(
            f"\nEvidence {displayed + 1}"
        )

        print(
            f"Source: {row['video_name']}"
        )

        print(
            f"Timestamp: "
            f"{row['start_time']:.2f}s → "
            f"{row['end_time']:.2f}s"
        )

        print(
            f"Relevance: "
            f"{row['retrieval_score']:.3f}"
        )

        display(
            Video(
                str(output_path),
                embed=True,
                width=640
            )
        )

        displayed += 1